### Structured Outputs

#### Pydantic

In [9]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = init_chat_model("google_genai:gemini-3.5-flash-lite")
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0', 'langchain-google-genai': '4.4.3'}}, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x000001C550FB0230>, default_metadata=(), model_kwar

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description= "The year the movie was released")
    director:str = Field(description = "The director of the movie")
    ratings:float = Field(description = "The Movie's Ratings out of 10")

In [ ]:
## Without Structred OP

model.invoke("Give me info on Movie Avengers:endgame")

AIMessage(content=[{'type': 'text', 'text': '**Avengers: Endgame** is a monumental superhero film released in 2019 by Marvel Studios. It serves as the direct sequel to *Avengers: Infinity War* (2018) and is the 22nd film in the Marvel Cinematic Universe (MCU). \n\nHere is a comprehensive breakdown of the movie:\n\n### 1. The Premise (No Spoilers for the Opening)\nThe movie picks up in the devastating aftermath of *Infinity War*, where the tyrannical alien warlord Thanos used the Infinity Stones to snap his fingers, wiping out 50% of all life in the universe (an event known as "The Blip"). The surviving Avengers—broken, grieving, and desperate—must figure out a way to undo Thanos\'s actions and restore the universe.\n\n### 2. Key Details\n* **Directors:** Anthony and Joe Russo\n* **Writers:** Christopher Markus and Stephen McFeely\n* **Release Date:** April 26, 2019 (US)\n* **Run Time:** 3 hours and 1 minute\n* **Budget:** ~$356 million\n* **Box Office:** It grossed approximately **$2.7

In [12]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0', 'langchain-google-genai': '4.4.3'}}, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x000001C550FB0230>, default

In [14]:
## With Structured OP

response = model_with_structure.invoke("Give me info on Movie Avengers:endgame")
response

Movie(title='Avengers: Endgame', year=2019, director='Anthony and Joe Russo', ratings=8.4)

### Message OP along side Parsed Structure

In [17]:
class Movie(BaseModel):
    """A Movie with details"""
    title:str = Field(..., description="The title of the movie")
    year:int = Field(..., description= "The year the movie was released")
    director:str = Field(..., description = "The director of the movie")
    ratings:float = Field(..., description = "The Movie's Ratings out of 10")

model_with_structure =  model.with_structured_output(Movie, include_raw = True)

response = model_with_structure.invoke("Give me info on Movie Avengers:Infinity war")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Avengers: Infinity War",\n  "year": 2018,\n  "director": "Anthony and Joe Russo",\n  "ratings": 8.4\n}', 'extras': {'signature': 'EmAKXgFpFH0TZLBMIboJI6yPgjH8uUYdwA0icMUGLIjWmawuTfl8QK72O/vUGsRl4D4OyDUxDL69h6TZ382vPiXy6z3Ok580BXSDMBwGJwqq4xdD/pMkFwfL4YyyqG/EG3Q='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e021-ed13-7298-9fa5-fe63f94fc9fa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 45, 'total_tokens': 55, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Avengers: Infinity War', year=2018, director='Anthony and Joe Russo', ratings=8.4),
 'parsing_error': None}

### Nested Structure

In [19]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    """Details of a Movie"""
    title:str 
    year:int 
    cast: list[Actor]
    genres: list[str]
    bugdet: float | None = Field(None, description="Budget in millions USD")
    
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about movie Mirage")
response

MovieDetails(title='Mirage', year=2018, cast=[Actor(name='Adriana Ugarte', role='Vera Roy'), Actor(name='Chino Darín', role='Inspector Leyra'), Actor(name='Javier Gutiérrez', role='Ángel Prieto')], genres=['Mystery', 'Drama', 'Sci-Fi', 'Thriller'], bugdet=None)

### TypedDict

In [30]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A Movie with details"""
    title : Annotated[str, ..., "The title of the movie"]
    year : Annotated[int, ...,  "The year the movie was released"]
    director : Annotated[str, ..., "The director of the movie"]
    ratings: Annotated[float, ..., "The Movie's Ratings out of 10"]

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("hey give me details of movie avengers")
response


{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'ratings': 8.0}

In [31]:
response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'ratings': 8.0}

In [32]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    """Details of a Movie"""
    title:str 
    year:int 
    cast: list[Actor]
    genres: list[str]
    bugdet: float | None = Field(None, description="Budget in millions USD")
    
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about movie Brand New Day")
response

{'title': 'Brand New Day',
 'year': 2015,
 'cast': [{'name': 'John Doe', 'role': 'Lead Actor'},
  {'name': 'Jane Smith', 'role': 'Supporting Actress'}],
 'genres': ['Drama', 'Romance'],
 'bugdet': 1500000.0}

In [33]:
model_with_typedict.profile

AttributeError: 'RunnableSequence' object has no attribute 'profile'

In [34]:
model.profile

{'name': 'Gemini 3.5 Flash Lite',
 'release_date': '2026-07-21',
 'last_updated': '2026-07-21',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'minimal'}

### DataClasses

In [35]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    "Contact info of a person"
    name: str =  Field(description="Name of the person")
    email: str = Field(description="Email of the person")
    phone: str = Field(description="Phone number of the person")

agent = create_agent(
    model = "google_genai:gemini-3.5-flash-lite",
    response_format = ContactInfo # Auto selects ProviderStratergy
)

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract contact info from: John Doe, john@example.com, 555 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, 555 123-4567', additional_kwargs={}, response_metadata={}, id='ff91e4db-bb53-46f6-90d1-59445d6672ef'),
  AIMessage(content=[{'type': 'text', 'text': '{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "555 123-4567"\n}', 'extras': {'signature': 'EmAKXgFpFH0Tg8/XG8pH9a9Qq+LyPHm3Ytd/gVdc9XEXsLQH/xmKr1OE9BedyRFljBqMGBSbcvEk7KoBQOuJNINtUtmNlYImkvxT18BPt9dapRGq43rlCRACGuRVvV14IAE='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e044-eb65-7075-80ac-4269f7f129f4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 43, 'total_tokens': 71, 'input_token_details': {'cache_read': 0}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='555 123-4567')}

In [36]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='555 123-4567'


In [ ]:
## dataclass

from dataclasses import dataclass

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "google_genai:gemini-3.5-flash-lite",
    response_format = ContactInfo # Auto selects ProviderStratergy
)

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract contact info from: John Doe, john@example.com, 555 123-4567"}]
})

result

### Same as before